# brAIn — Brain Tumor Detection Pipeline with LLM Integration

**Mustafa Subhani · Arnau Rey**

A walkthrough of the full pipeline on a real MRI scan, using the actual
running application — every screenshot below is either a real capture of
the app in operation, or (where noted) a higher-quality capture from the
project's own paper / presentation, taken when the system was running on a
GPU rather than the CPU-only environment used for this repo. Nothing here
is mocked or hand-edited for content — only two images were lightly cropped
to remove UI elements (an old Cross-Scanner toggle) that no longer exist in
the current app.

> **Research / educational project.** Not a medical device, not validated
> by any regulatory authority, not for clinical decision-making.

See [`README.md`](../README.md) for full setup instructions, architecture,
dataset details, and results tables — this notebook focuses on *what the
system actually does*, end-to-end, on one scan.


## 1. Motivation

A prior version of this project was a binary ResNet-50 classifier —
"tumor" vs. "no tumor" — reaching ~99.3% accuracy. That's a reasonable
screening signal, but it stops short of being useful to a specialist:
knowing a tumor is *present* matters far less in practice than knowing what
*type* it likely is, roughly *where* it sits, how *confident* the model
actually is, and *which part of the image* drove that answer.

This project extends that binary detector into a full diagnostic-support
pipeline:

- **4-class classification** (glioma / meningioma / pituitary / no tumor) via a 3-model CNN ensemble
- **Uncertainty quantification** (MC Dropout) and **out-of-distribution detection**, so the system can say "I'm not sure" instead of guessing confidently
- **Visual explainability** (4-level hierarchical Grad-CAM / LayerCAM), so a clinician can see *where* the model is looking
- **Supervised tumor localization** (YOLO11n + MobileSAM), independent of classifier attention
- **An LLM-generated diagnostic report** (MedGemma 1.5 4B) with a conversational follow-up chat

Related work in this space includes the BraTS segmentation challenge,
**DISCERN** (Vall d'Hebron/VHIO/IDIBELL — differentiates 3 malignant brain
tumor types from MRI at ~78% accuracy, a useful reference for how hard this
problem is), and a UPM/CIBER-BBN glioma segmentation system. Regulatory
guidance (FDA and others) is explicit that clinical AI needs to demonstrate
transparency and interpretability, not just raw accuracy — which is the
whole reason this pipeline surfaces uncertainty, disagreement, and visual
explanations instead of a bare label.


## 2. Pipeline overview

```
   MRI input
       │
       ▼
 Preprocessing (brain extraction, CLAHE)
       │
   ┌───┴────────────────────────┐
   ▼                            ▼
CNN ensemble (×3, TTA×5)    YOLO11n + MobileSAM
+ MC Dropout uncertainty     (tumor bbox + mask)
+ energy-based OOD score            │
+ hierarchical Grad-CAM/LayerCAM    │
   │                                │
   └──────────────┬─────────────────┘
                   ▼
          MedGemma 1.5 4B
   (independent size cross-check,
    diagnostic report, chat)
                   │
                   ▼
          React UI + 3D brain atlas
```

The sections below follow this exact order: preprocessing → training-time
data augmentation → the Analysis tab (classification through report
generation) → Metrics tab → 3D Brain Atlas.


## 3. Preprocessing

Before anything reaches the classifiers, each upload goes through brain
extraction and **CLAHE** (Contrast-Limited Adaptive Histogram Equalization)
— the same normalization recipe applied during training, so inference-time
images match the distribution the models actually learned from. On a real
meningioma scan, CLAHE visibly recovers soft-tissue contrast that a flat
histogram would leave washed out:

![CLAHE before/after on a real meningioma scan](assets/preprocessing/01_clahe_before_after.png)


## 4. Training-time data augmentation

Every one of the 9,867 training images is re-augmented independently, *on
the fly*, every epoch — nothing augmented is ever written to disk. Over a
40-epoch training run that works out to **394,680 distinct stochastic
views** of the training set (9,867 × 40), averaging roughly 2.5
transformations applied per view:

| Transform | Probability | Views over 40 epochs | Why |
|---|---:|---:|---|
| Horizontal flip | 0.50 | ≈197,340 | Tumor type doesn't depend on left/right side |
| Rotation ±15° | 0.50 | ≈197,340 | Realistic range; larger angles break radiological convention |
| Brightness/contrast ±20% | 0.50 | ≈197,340 | Robustness to exposure differences between acquisitions |
| CLAHE | 0.50 | ≈197,340 | Matches the inference-time preprocessing exactly |
| Gaussian noise (σ=0.01) | 0.30 | ≈118,404 | Simulates acquisition noise without erasing real detail |
| Elastic transform (mild) | 0.20 | ≈78,936 | Simulates inter-scanner geometric variability |

Deliberately **not** applied: vertical flip (superior/inferior anatomical
direction is meaningful, unlike left/right) and strong color perturbation
(T1 MRI is essentially grayscale, so hue/saturation jitter wouldn't
generalize to anything real). Source: `app/src/train_v2.py` →
`build_train_transform()`.


## 5. Analysis tab — full walkthrough

### 5.1 Upload

Drag-and-drop or click to upload an MRI slice. The panel below is a fresh
capture from the current app (Cross-Scanner mode has since been removed —
"Standard Scanner" is the only mode now).

![Upload panel](assets/analysis/00_upload_panel.png)

### 5.2 Three-model ensemble agreement

Every scan is classified independently by three CNNs (ConvNeXt-Tiny,
EfficientNet-B3, ResNet-50), each with test-time augmentation (TTA×5). All
three have to actually agree before the app presents a confident result —
this run, all three unanimously predict **Meningioma**. The
highest-confidence model also reports MC-Dropout uncertainty from 20
stochastic passes, decomposed into epistemic (model uncertainty) and
aleatoric (data ambiguity) components.

![Model comparison — 3/3 unanimous](assets/analysis/01_model_comparison.png)

### 5.3 Trust verdict — flagged for review

Even with a confident, unanimous prediction, the app runs independent
consistency checks — uncertainty threshold, focus-crop re-classification
(does the model still agree when shown only its own attention region?),
out-of-distribution score — and flags the scan for expert review if any of
them trip, rather than only ever showing green when the classifier sounds
confident.

![Caution banner — multiple flags](assets/analysis/02_caution_banner.png)

### 5.4 Malignancy assessment

Combines the classifier's output with an independently computed tumor size
(YOLO11n bounding box + MobileSAM pixel-tight segmentation) and anatomical
location into a 0–10 malignancy score. Two size estimates are shown
side-by-side — the pixel-based pipeline's and MedGemma's own independent
visual estimate from the raw image — so a disagreement between them is
visible rather than silently resolved into one number.

![Malignancy assessment with tumor bounding box](assets/analysis/03_malignancy_assessment.png)

### 5.5 Symptom-aware scoring + clinical context

Users can optionally tap any symptoms they've noticed, which adjusts the
malignancy score transparently (shown as a visible bonus, not hidden inside
a black box). Below that, reference clinical context for the predicted
tumor type — static medical knowledge, not LLM-generated, so it reads
identically every time.

![Symptoms and clinical context](assets/analysis/04_symptoms_clinical_context.png)

The same clinical-context panel in Spanish, showing the app's full EN/ES
toggle in practice — every report, chat response, and UI label translates,
not just static strings:

![Clinical context, Spanish](assets/analysis/04b_clinical_context.png)

### 5.6 Doctor-visit prep + multi-color anatomical views

A ready-to-use list of follow-up questions for the specific tumor type,
plus the same MRI slice re-rendered in several color maps.

![Questions for your doctor + anatomy color schemes](assets/analysis/05_questions_anatomy.png)

Different colormaps (HOT, JET, BONE, VIRIDIS, and more) make different
tissue contrasts visible that a single grayscale view can hide — useful
when comparing subtle density differences at the tumor margin:

![Six-colormap anatomical grid](assets/analysis/05b_anatomy_colors.png)

### 5.7 Hierarchical 4-level explainability

Four increasingly specific heat-maps answering four different questions
about the same prediction — Grad-CAM (*is there a tumor?*), Grad-CAM++
(*what type?*), LayerCAM block4 (*where exactly?*), and fused LayerCAM
(*how does the model's reasoning combine across layers?*). All four
consistently highlight the same lesion, which is itself a sanity check — if
they didn't agree with each other, that would suggest the classifier's
attention isn't actually anchored on the tumor.

![Hierarchical 4-level XAI](assets/analysis/06_hierarchical_xai.png)

### 5.8 MedGemma diagnostic report

MedGemma 1.5 4B (served locally via Ollama) generates a structured
diagnostic report from the prediction, confidence, and tumor
characteristics — Basic or Advanced detail level, in English or Spanish. On
CPU-only inference this genuinely takes a few minutes per report (the model
frequently burns tokens on internal chain-of-thought before emitting the
formatted output; the app has retry and partial-recovery logic for this —
see the README's troubleshooting section).

English report, Glioma case:

![MedGemma diagnostic report, English](assets/analysis/07_medgemma_report.png)

Spanish report, Meningioma case:

![MedGemma diagnostic report, Spanish](assets/analysis/07b_medgemma_report_es.png)


## 6. Metrics tab

Model comparison and ensemble-agreement statistics, computed once on the
locked 2,114-image test set and served directly from
`reports/v2_metrics.json` (not recomputed live — this is exactly what
`app/src/eval_v2.py` produced):

![Metrics overview — accuracy, F1, latency, ensemble agreement](assets/metrics/01_overview.png)

Per-model confusion matrices plus the soft-vote ensemble's — the diagonal
is correct classifications, and comparing the three individual panels
against the fourth ensemble panel is what makes voting's error-correction
effect visible directly, rather than just as a summary number:

![Confusion matrices — 3 models + ensemble](assets/metrics/02_confusion_matrices.png)


## 7. 3D Brain Atlas

The predicted tumor's anatomical region links out to an interactive 3D
brain model (11 labeled regions), so "frontal lobe, left hemisphere" is
something a patient can actually see rather than just read:

![Interactive 3D brain atlas](assets/atlas/01_3d_view.png)

The atlas view links back into the MedGemma report/chat for that same
region, keeping the anatomical and narrative explanations connected instead
of living in two disconnected tabs:

![3D atlas linked to MedGemma report](assets/atlas/02_medgemma_link.png)


## 8. Limitations & future work

This is an experimental support tool, not a validated clinical system:

- **Data diversity** — currently axial T1 slices from three public
  datasets; real clinical use would need BraTS 2023/2024, anonymized
  hospital data, multiple sequences (T1c/T2/FLAIR), and per-source
  cross-dataset validation to catch domain shift.
- **No clinical validation yet** — no radiologist has compared the
  system's output against expert judgment.
- **Localization, not segmentation** — YOLO11n + MobileSAM give a
  bounding box and pixel-tight mask, not a clinically-validated
  segmentation; a natural next step is fine-tuning MedSAM (possibly via
  LoRA) and evaluating with Dice/IoU/Hausdorff against expert annotations.
- **No automatic MRI-sequence detection** yet, to catch a wrong sequence
  before it silently degrades the analysis.
- **Single VLM** — MedGemma 1.5 4B hasn't been benchmarked against
  alternatives (e.g. Qwen3-VL) for report quality or stability.
- **Local-only, single-user** — no encryption, access control, or
  auto-deletion; a real deployment would need GDPR/HIPAA-aware handling
  from the start.

Full detail on each of these, plus references and dataset citations, is in
the [main README](../README.md#11-limitations--future-work).
